In [1]:
import tkinter as tk
from tkinter import filedialog
import pandas as pd
import os
import fnmatch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.widgets import LassoSelector
from scipy import signal
import numpy as np
%matplotlib qt

In [2]:
# Initialize tkinter root
root = tk.Tk()
root.withdraw()  # Hide the root window
root.attributes("-topmost", True)

# Define the initial directory
initial_directory = r"C:\Users\fossati.veronica\Desktop\Rewire_Project-master\Rewire_Project-master\Code\mediapipe\Data"  # Change this to your desired folder

#select one file for peak identification
file_path = filedialog.askopenfilename(
    initialdir=initial_directory, 
    filetypes=[("Filtered Csv Files", "*_mod*.csv")], 
    title="Select a Text File"
)

base_dir, file_name = os.path.split(file_path)
all_files = os.listdir(base_dir)
f1 = base_dir + "/"
f1 = f1 + fnmatch.filter(all_files, f"*{file_name.split('_')[1]}-startTime*.txt")[0]
f2 = base_dir + "/"
f2 = f2 + fnmatch.filter(all_files, f"*{file_name.split('_')[1]}_stimParams*.txt")[0]

data_kinematics = pd.read_csv(file_path)

with open(f1) as file:
    start_time = [line.rstrip() for line in file]
with open(f2) as file:
    stims = [line.rstrip() for line in file]
stim_events=[]
stim_type=[]
stim_params=[]

date_format = "%Y-%m-%d %H:%M:%S.%f"

for i_s,s in enumerate(stims):
    if 'min freq' in s:
        stim_events.append(pd.Timestamp(f'{os.path.split(f1)[1][:10]} '+stims[i_s-1])-pd.Timedelta(seconds=1))  #MODIFY with different timepoint
        stim_type.append('freq')
        stim_params.append((int(stims[i_s+1].split(',')[4]),int(stims[i_s+1].split(',')[5]),int(stims[i_s+1].split(',')[6])))
    elif 'min amp' in s:
        stim_events.append(pd.Timestamp(f'{os.path.split(f1)[1][:10]} '+stims[i_s-1])-pd.Timedelta(seconds=1))
        stim_type.append('amp')
        stim_params.append((int(stims[i_s+1].split(',')[2]),int(stims[i_s+1].split(',')[3]),int(stims[i_s+1].split(',')[4])))

In [3]:
data_kinematics

,Unnamed: 0,Thumb,Index,Middle,Ring,Pinky
0,2024-10-18 11:10:52.023900,137.246695,129.385492,106.761637,105.916776,123.413588
1,2024-10-18 11:10:52.055933,137.240811,129.496256,106.757569,105.878043,123.317747
2,2024-10-18 11:10:52.102500,137.229299,129.605220,106.753025,105.840621,123.219030
3,2024-10-18 11:10:52.166633,137.211880,129.710083,106.748064,105.805137,123.119552
4,2024-10-18 11:10:52.199967,137.188473,129.808702,106.742806,105.772233,123.021531
...,...,...,...,...,...,...
1383,2024-10-18 11:11:41.143900,132.978169,129.629066,105.991557,104.745147,122.360024
1384,2024-10-18 11:11:41.190567,132.905150,129.806988,106.022735,104.735606,122.246971
1385,2024-10-18 11:11:41.223900,132.838748,130.003027,106.054046,104.720961,122.127524
1386,2024-10-18 11:11:41.255800,132.779419,130.211340,106.084922,104.702148,122.004887


In [4]:
fig, axs = plt.subplots(len(data_kinematics.columns), 1, figsize=(10, 12), sharex=True)



#create stimulation pattern for plotting
time = []
for i in data_kinematics[data_kinematics.columns[0]].values:
    time.append(pd.Timestamp(i))
time_series=pd.Series(time)

stim_wave = []
timestamps=time_series[time_series < stim_events[0]]
timestamps = timestamps.tolist()
for ts in timestamps:
    stim_wave.append(0)

vertical_lines=[]
for i_stim,stim in enumerate(stim_params):
    n = 9   #to be changed properly
    trials=3
    current_value = 1  # Starting value for the step function
    steps=int((stim[1]-stim[0])/stim[2])+1

    if i_stim==0:
        timestamps = time_series[(time_series > stim_events[0]) & (time_series < stim_events[0]+pd.Timedelta(seconds=n*steps))]
    else:
        timestamps = time_series[(time_series > stim_events[i_stim]) & (time_series < stim_events[i_stim]+pd.Timedelta(seconds=n*steps))]
    
    timestamps = timestamps.tolist()
    next_change_time =  timestamps[0] + pd.Timedelta(seconds=n)
    vertical_lines.append(timestamps[0])
    
    for ts in timestamps:
        if ts >= next_change_time:
            vertical_lines.append(ts)
            current_value += 1  # Increment the step value at each interval
            next_change_time = ts + pd.Timedelta(seconds=n)  # Update to the next change time
        stim_wave.append(current_value)
    vertical_lines.append(timestamps[-1])

    if i_stim==len(stim_params)-1:
        timestamps=time_series[time_series > stim_events[i_stim]+pd.Timedelta(seconds=n*steps)]
    else:
        timestamps=time_series[(time_series > stim_events[i_stim]+pd.Timedelta(seconds=n*steps)) & (time_series < stim_events[i_stim+1])]
    timestamps=timestamps.tolist()
    for ts in timestamps:
        stim_wave.append(0)


#initialize GUI for peak identification: right button --> set threshold, left button and move --> move points, +/- keys --> add/remove points
def onselect(verts, i):

    threshold = verts[-1][1]   
    th_line[i].set_data(time_for_plot, [threshold]*len(time))
    
    # to find peaks based on the selected height
    peaks_temp, props = signal.find_peaks(x=-data_kinematics[data_kinematics.columns[i+1]].values, height=-threshold, distance=3*24)
    peaks_temp = np.array(peaks_temp).astype(int)
    peaks_points[i].set_data(np.array(time_for_plot)[peaks_temp], data_kinematics[data_kinematics.columns[i+1]].values[peaks_temp])
    
    fig.canvas.draw_idle() 

def on_press(event):
    global press
    global index
    global closest_point_pressed
    
    if not any(event.inaxes == axis for axis in axs):
        return

    index = None
    for i, axis in enumerate(axs):
        if event.inaxes == axis:
            index = i-1
            break

    contains_point, attrd = peaks_points[index].contains(event)
    xdata_point = peaks_points[index].get_xdata()
    ydata_point = peaks_points[index].get_ydata()
    
    if not contains_point or index<0:
        return

    event_pixel_coords = axs[index+1].transData.transform((event.xdata, event.ydata))
    event_x_pixel, event_y_pixel = event_pixel_coords
    data_pixel_coords = axs[index+1].transData.transform(np.vstack([xdata_point, ydata_point]).T)
    x_pixel = data_pixel_coords[:, 0]
    y_pixel = data_pixel_coords[:, 1]
    distances = np.sqrt((x_pixel - event_x_pixel)**2 + (y_pixel - event_y_pixel)**2)
    if len(distances) > 0:
        closest_point_pressed = np.argmin(distances)
        press = (xdata_point[closest_point_pressed], ydata_point.astype('float64')[closest_point_pressed]), (event.xdata, event.ydata)
        
def on_motion(event):

    global press
    global index
    global closest_point_pressed

    if press is None or not any(event.inaxes == axis for axis in axs) or index<0:
        return
        
    (x0, y0), (xpress, ypress) = press
    dx = event.xdata - xpress
    dy = event.ydata - ypress

    xdata = peaks_points[index].get_xdata()
    ydata = peaks_points[index].get_ydata()
    idx = np.argmin(np.abs(lines[index].get_xdata()-(x0+dx)))
    x_new = lines[index].get_xdata()[idx]
    y_new = lines[index].get_ydata()[idx]
    xdata[closest_point_pressed] = x_new
    ydata[closest_point_pressed] = lines[index].get_ydata()[idx]
    peaks_points[index].set_data(xdata, ydata)
            
    fig.canvas.draw()

def on_release(event):
    
    global press
    global closest_point_pressed
    global index
    press = None
    closest_point = None

    sort_points(peaks_points[index])
    index=None
    fig.canvas.draw()

def sort_points(points):
    xdata = points.get_xdata()
    ydata = points.get_ydata()
    if len(xdata)>0:
        idx_sort = np.argsort(xdata)
        xdata=xdata[idx_sort]
        ydata=ydata[idx_sort]
        points.set_data(xdata, ydata)

def on_key_press(event):
    if not any(event.inaxes == axis for axis in axs):
        return

    index = None
    for i, axis in enumerate(axs):
        if event.inaxes == axis:
            index = i-1
            break
    
    if event.key == '+':
        xdata = lines[index].get_xdata()
        ydata = lines[index].get_ydata()
        event_pixel_coords = axs[index+1].transData.transform((event.xdata, event.ydata))
        event_x_pixel, event_y_pixel = event_pixel_coords
        data_pixel_coords = axs[index+1].transData.transform(np.vstack([xdata, ydata]).T)
        x_pixel = data_pixel_coords[:, 0]
        y_pixel = data_pixel_coords[:, 1]
        distances = np.sqrt((x_pixel - event_x_pixel)**2 + (y_pixel - event_y_pixel)**2)
        if len(distances) > 0:
            mindist = np.min(distances)
            if mindist > 5:
                return
            closest_point = np.argmin(distances)
            x_new = xdata[closest_point]
        peaks_points[index].set_data(np.append(peaks_points[index].get_xdata(), x_new), np.append(peaks_points[index].get_ydata(), ydata[closest_point]))
        sort_points(peaks_points[index])

    elif event.key == '-':
        xdata_points = peaks_points[index].get_xdata()
        ydata_points = peaks_points[index].get_ydata()
        event_pixel_coords = axs[index+1].transData.transform((event.xdata, event.ydata))
        event_x_pixel, event_y_pixel = event_pixel_coords
        data_pixel_coords = axs[index+1].transData.transform(np.vstack([xdata_points, ydata_points]).T)
        x_pixel = data_pixel_coords[:, 0]
        y_pixel = data_pixel_coords[:, 1]
        distances = np.sqrt((x_pixel - event_x_pixel)**2 + (y_pixel - event_y_pixel)**2)
        if len(distances) > 0:
            closest_point = np.argmin(distances)
            if distances[closest_point] < 6:  # Tolerance for click distance 
                xdata_points = np.delete(xdata_points, closest_point)
                ydata_points = np.delete(ydata_points, closest_point)
                peaks_points[index].set_data(xdata_points, ydata_points)
    fig.canvas.draw()

#initialize plot
time_for_plot=(mdates.date2num(time)-mdates.date2num(time)[0])
vertical_lines_for_plot=(mdates.date2num(vertical_lines)-mdates.date2num(time)[0])
peaks_points=[]
th_line=[]
lines=[]
for i, finger in enumerate(data_kinematics.columns):
    if i==0:
        axs[i].plot(time_for_plot,stim_wave)
        
    else:
        data_plot=data_kinematics[finger].values
        line,=axs[i].plot(time_for_plot,data_plot, label=finger, color='#9d0208') #
        lines.append(line)
        
        rec_curves_points, = axs[i].plot([],[], 'bo', markeredgecolor='black')
        peaks_points.append(rec_curves_points)

        ll, = axs[i].plot(time_for_plot, [data_kinematics[finger][0]]*len(time), 'r--', alpha=1, linewidth=0.5)
        th_line.append(ll)
        
        axs[i].set_title(finger)
        #axs[i].set_xlim((vertical_lines_for_plot[0],time_for_plot[-1]))

        for l in vertical_lines_for_plot:
            axs[i].axvline(l)

lasso_thumb = LassoSelector(axs[1], lambda verts: onselect(verts, i=0), button=3)
lasso_index = LassoSelector(axs[2], lambda verts: onselect(verts, i=1), button=3)
lasso_medium = LassoSelector(axs[3], lambda verts: onselect(verts, i=2), button=3)
lasso_ring = LassoSelector(axs[4], lambda verts: onselect(verts, i=3), button=3)
lasso_pinky = LassoSelector(axs[5], lambda verts: onselect(verts, i=4), button=3)

for i in range(5):
    peaks_points[i].set_picker(5)
    cidpress_hs = peaks_points[i].figure.canvas.mpl_connect(
        'button_press_event', on_press)
    cidrelease_hs = peaks_points[i].figure.canvas.mpl_connect(
        'button_release_event', on_release)
    cidmotion_hs = peaks_points[i].figure.canvas.mpl_connect(
        'motion_notify_event', on_motion)

fig.canvas.mpl_connect('key_press_event', on_key_press)

index=None
press=None
closest_point_pressed=None




figManager = plt.get_current_fig_manager()
figManager.window.showMaximized()

In [5]:
peaks_kinematics_abs=dict()
peaks_kinematics_common_ref=dict()
peaks_kinematics_single_ref=dict()
stim_vect=np.arange(stim_params[0][0], stim_params[0][1]+1, stim_params[0][2])
ref_common=[]

#compute peaks
for i, finger in enumerate(data_kinematics.columns[1:]):
    
    peaks_kinematics_abs[finger]=dict()  #absolute values of the angles reached during contraction
    peaks_kinematics_common_ref[finger]=dict()  #reference position is computed as mean of 2 secs before stim init
    peaks_kinematics_single_ref[finger]=dict() #single reference for each peak, as the resting position before each flexion
    
    peaks_x=peaks_points[i].get_xdata()
    peaks_y=peaks_points[i].get_ydata()
    peaks_y=np.delete(peaks_y, np.where(peaks_x<vertical_lines_for_plot[0]))
    peaks_x=np.delete(peaks_x, np.where(peaks_x<vertical_lines_for_plot[0]))
    peaks_y=np.delete(peaks_y, np.where(peaks_x>vertical_lines_for_plot[-1]))
    peaks_x=np.delete(peaks_x, np.where(peaks_x>vertical_lines_for_plot[-1]))

    single_ref=[]
    for el in peaks_x:
        idx_start=np.where(time_for_plot==el)[0]
        #print(idx_start)
        for aaa in np.arange(idx_start-1,0,-1):
            if data_kinematics[finger].values[aaa]-data_kinematics[finger].values[aaa-1]>=0:
                #print(aaa)
                single_ref.append(data_kinematics[finger].values[aaa])
                break
    ref_common.append(np.mean(data_kinematics[finger].values[np.where((time_for_plot>vertical_lines_for_plot[0]-2/(24*60*60)) & (time_for_plot<vertical_lines_for_plot[0]))]))

    for i_l, l in enumerate(vertical_lines_for_plot[:-1]):
        peaks_kinematics_abs[finger][stim_vect[i_l]] = peaks_y[np.where((peaks_x>l) & (peaks_x<vertical_lines_for_plot[i_l+1]))]
        peaks_kinematics_single_ref[finger][stim_vect[i_l]] = -peaks_y[np.where((peaks_x>l) & (peaks_x<vertical_lines_for_plot[i_l+1]))]+np.array(single_ref)[np.where((peaks_x>l) & (peaks_x<vertical_lines_for_plot[i_l+1]))]
        if peaks_kinematics_abs[finger][stim_vect[i_l]].shape[0]<3:
            peaks_kinematics_single_ref[finger][stim_vect[i_l]] = np.concatenate((peaks_kinematics_single_ref[finger][stim_vect[i_l]],np.repeat(0,3-peaks_kinematics_abs[finger][stim_vect[i_l]].shape[0])))
            peaks_kinematics_abs[finger][stim_vect[i_l]] = np.concatenate((peaks_kinematics_abs[finger][stim_vect[i_l]],np.repeat(0,3-peaks_kinematics_abs[finger][stim_vect[i_l]].shape[0])))
            

        peaks_kinematics_common_ref[finger][stim_vect[i_l]]=np.empty((3,))
        for kk in range(3):
            if peaks_kinematics_abs[finger][stim_vect[i_l]][kk]==0:
                peaks_kinematics_common_ref[finger][stim_vect[i_l]][kk]=0.0
            else:
                peaks_kinematics_common_ref[finger][stim_vect[i_l]][kk]=-peaks_kinematics_abs[finger][stim_vect[i_l]][kk]+ref_common[i]
                if peaks_kinematics_common_ref[finger][stim_vect[i_l]][kk]<0:
                    peaks_kinematics_common_ref[finger][stim_vect[i_l]][kk]=0


C:\Users\fossati.veronica\AppData\Local\Temp\ipykernel_27712\4027020429.py:25: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  for aaa in np.arange(idx_start-1,0,-1):


In [6]:
peaks_kinematics_abs

{'Thumb': {190: array([0., 0., 0.]),
  200: array([120.11090399, 118.55950744, 116.57998843]),
  210: array([107.86919876, 105.52137969, 110.34888237]),
  220: array([104.74061859, 106.60618129, 103.63384825])},
 'Index': {190: array([0., 0., 0.]),
  200: array([118.02280119, 119.53186166, 113.46959879]),
  210: array([90.9123714 , 91.73477972, 91.14939299]),
  220: array([75.58561652, 69.98044322, 69.67831751])},
 'Middle': {190: array([0., 0., 0.]),
  200: array([95.10857151, 93.98412458, 91.95530842]),
  210: array([86.50691552, 87.6295097 , 87.76994647]),
  220: array([73.15771226, 68.91517286, 68.48666843])},
 'Ring': {190: array([0., 0., 0.]),
  200: array([92.93871826, 90.96587196, 89.8822887 ]),
  210: array([85.74738721, 86.71273881, 86.41498883]),
  220: array([70.24343922, 68.41389913, 67.1206569 ])},
 'Pinky': {190: array([0., 0., 0.]),
  200: array([117.70901949, 118.4790179 , 118.1673907 ]),
  210: array([116.97425705, 116.58381127, 117.86895428]),
  220: array([115.61344

In [7]:
peaks_kinematics_common_ref

{'Thumb': {190: array([0., 0., 0.]),
  200: array([15.15285023, 16.70424678, 18.68376579]),
  210: array([27.39455546, 29.74237452, 24.91487184]),
  220: array([30.52313562, 28.65757293, 31.62990596])},
 'Index': {190: array([0., 0., 0.]),
  200: array([11.06408627,  9.5550258 , 15.61728867]),
  210: array([38.17451607, 37.35210774, 37.93749447]),
  220: array([53.50127094, 59.10644424, 59.40856995])},
 'Middle': {190: array([0., 0., 0.]),
  200: array([11.20291054, 12.32735747, 14.35617363]),
  210: array([19.80456653, 18.68197235, 18.54153558]),
  220: array([33.15376979, 37.39630919, 37.82481362])},
 'Ring': {190: array([0., 0., 0.]),
  200: array([12.41681524, 14.38966154, 15.4732448 ]),
  210: array([19.60814628, 18.64279469, 18.94054467]),
  220: array([35.11209428, 36.94163437, 38.2348766 ])},
 'Pinky': {190: array([0., 0., 0.]),
  200: array([4.77447558, 4.00447717, 4.31610437]),
  210: array([5.50923801, 5.8996838 , 4.61454079]),
  220: array([ 6.87005229,  9.70219643, 10.5864

In [8]:
peaks_kinematics_single_ref

{'Thumb': {190: array([0., 0., 0.]),
  200: array([15.64096927, 17.23562782, 18.7207901 ]),
  210: array([28.03249233, 30.24618717, 25.47101303]),
  220: array([30.79919157, 28.1382952 , 30.63405954])},
 'Index': {190: array([0., 0., 0.]),
  200: array([12.04801151, 10.06506728, 15.84372862]),
  210: array([39.66071169, 38.16661655, 38.19660646]),
  220: array([52.99281816, 60.49542602, 59.72790334])},
 'Middle': {190: array([0., 0., 0.]),
  200: array([12.11946351, 13.46860252, 15.17692289]),
  210: array([20.73769289, 19.34904838, 18.96082254]),
  220: array([33.18416791, 37.17144624, 37.26418939])},
 'Ring': {190: array([0., 0., 0.]),
  200: array([13.88110122, 16.61045406, 16.83359891]),
  210: array([20.65435859, 20.18923565, 20.05741667]),
  220: array([35.85608328, 36.24414934, 37.11709402])},
 'Pinky': {190: array([0., 0., 0.]),
  200: array([5.69633033, 5.42086868, 5.01442897]),
  210: array([5.90321849, 6.74582633, 5.97653244]),
  220: array([7.22871109, 9.31890232, 9.6533533

In [12]:
file_path_r=base_dir + "/" + file_name[11:15] + ".npy"
np.save(file_path_r, [peaks_kinematics_single_ref, peaks_kinematics_common_ref, peaks_kinematics_abs])